In [16]:
# Source - https://stackoverflow.com/a/43615015
# Posted by johndodo, modified by community. See post 'Timeline' for change history
# Retrieved 2026-10-19, License - CC BY-SA 4.0

%run anamoly_pipeline.ipynb

____BAD ENTRIES____(for bad count greater than 1)
                 minute   bad  total
575 2026-10-01 09:35:00   2.0     74
845 2026-10-01 14:05:00  19.0     60
846 2026-10-01 14:06:00  14.0     54
847 2026-10-01 14:07:00  21.0     62
848 2026-10-01 14:08:00  19.0     52
849 2026-10-01 14:09:00  17.0     40
850 2026-10-01 14:10:00  12.0     52
851 2026-10-01 14:11:00  29.0     78
852 2026-10-01 14:12:00  34.0     88
853 2026-10-01 14:13:00  16.0     50
854 2026-10-01 14:14:00  35.0     92
855 2026-10-01 14:15:00  12.0     40
856 2026-10-01 14:16:00  24.0     68

MEDIAN: 0.0
MAD: 0.0

____SCORE_____(flagged)
                 minute   bad  total      score
845 2026-10-01 14:05:00  19.0     60  12.815324
846 2026-10-01 14:06:00  14.0     54   9.442871
847 2026-10-01 14:07:00  21.0     62  14.164306
848 2026-10-01 14:08:00  19.0     52  12.815324
849 2026-10-01 14:09:00  17.0     40  11.466343
850 2026-10-01 14:10:00  12.0     52   8.093889
851 2026-10-01 14:11:00  29.0     78  19.560232
8

In [7]:
import re

In [8]:
def _dt(x):
    return pd.Timestamp(x).to_pydatetime()


def service_summary(start, end):
    """Per-service error/warn counts in the window, ordered by who failed first."""
    df = query_df("""
        SELECT service,
               CAST(SUM(level = 'ERROR') AS SIGNED) AS errors,
               CAST(SUM(level = 'WARN') AS SIGNED) AS warns,
               COUNT(*) AS total_lines,
               MIN(CASE WHEN level = 'ERROR' THEN ts END) AS first_error,
               ROUND(AVG(latency_ms)) AS avg_latency_ms
        FROM logs
        WHERE ts >= :s AND ts < :e
        GROUP BY service
    """, s=_dt(start), e=_dt(end))
    # services with no errors (NaT) go last
    return df.sort_values("first_error", na_position="last").reset_index(drop=True)


def _template(msg: str) -> str:
    """Turn a message into a pattern so similar errors group together."""
    msg = re.sub(r"[0-9a-fA-F]{8}-[0-9a-fA-F-]{27}", "<uuid>", msg)
    msg = re.sub(r"for user \S+", "for user <user>", msg)
    msg = re.sub(r"\d+", "<n>", msg)
    return msg


def error_groups(start, end, top=10):
    """Group ERROR/WARN lines by (service, level, error_type, message pattern)."""
    df = query_df("""
        SELECT ts, service, level, error_type, message
        FROM logs
        WHERE level IN ('ERROR', 'WARN') AND ts >= :s AND ts < :e
    """, s=_dt(start), e=_dt(end))
    if df.empty:
        return df
    df["pattern"] = df["message"].map(_template)
    groups = (
        df.groupby(["service", "level", "error_type", "pattern"], dropna=False)
        .agg(count=("ts", "size"), first_seen=("ts", "min"), last_seen=("ts", "max"))
        .reset_index()
        .sort_values("count", ascending=False)
        .head(top)
    )
    return groups


In [10]:
period = find_anomalous_period(start, end)
print(period)

____BAD ENTRIES____(for bad count greater than 1)
                 minute   bad  total
575 2026-10-01 09:35:00   2.0     74
845 2026-10-01 14:05:00  19.0     60
846 2026-10-01 14:06:00  14.0     54
847 2026-10-01 14:07:00  21.0     62
848 2026-10-01 14:08:00  19.0     52
849 2026-10-01 14:09:00  17.0     40
850 2026-10-01 14:10:00  12.0     52
851 2026-10-01 14:11:00  29.0     78
852 2026-10-01 14:12:00  34.0     88
853 2026-10-01 14:13:00  16.0     50
854 2026-10-01 14:14:00  35.0     92
855 2026-10-01 14:15:00  12.0     40
856 2026-10-01 14:16:00  24.0     68

MEDIAN: 0.0
MAD: 0.0

____SCORE_____(flagged)
                 minute   bad  total      score
845 2026-10-01 14:05:00  19.0     60  12.815324
846 2026-10-01 14:06:00  14.0     54   9.442871
847 2026-10-01 14:07:00  21.0     62  14.164306
848 2026-10-01 14:08:00  19.0     52  12.815324
849 2026-10-01 14:09:00  17.0     40  11.466343
850 2026-10-01 14:10:00  12.0     52   8.093889
851 2026-10-01 14:11:00  29.0     78  19.560232
8

In [18]:
print(service_summary(start, end))

         service  errors  warns  total_lines             first_error  \
0     db-service      84      0        20000 2026-10-01 14:05:06.207   
1  order-service      84      0        20084 2026-10-01 14:05:09.247   
2    api-gateway      84      0        20084 2026-10-01 14:05:09.297   
3   auth-service       0     88        20088                     NaT   

   avg_latency_ms  
0            55.0  
1            55.0  
2            55.0  
3            42.0  


In [19]:
print(service_summary(start, end).to_dict("records"))

[{'service': 'db-service', 'errors': 84, 'warns': 0, 'total_lines': 20000, 'first_error': Timestamp('2026-10-01 14:05:06.207000'), 'avg_latency_ms': 55.0}, {'service': 'order-service', 'errors': 84, 'warns': 0, 'total_lines': 20084, 'first_error': Timestamp('2026-10-01 14:05:09.247000'), 'avg_latency_ms': 55.0}, {'service': 'api-gateway', 'errors': 84, 'warns': 0, 'total_lines': 20084, 'first_error': Timestamp('2026-10-01 14:05:09.297000'), 'avg_latency_ms': 55.0}, {'service': 'auth-service', 'errors': 0, 'warns': 88, 'total_lines': 20088, 'first_error': NaT, 'avg_latency_ms': 42.0}]


In [20]:
print(error_groups(start, end))

         service  level            error_type  \
1   auth-service   WARN          BAD_PASSWORD   
0    api-gateway  ERROR               TIMEOUT   
2     db-service  ERROR  DB_CONNECTION_FAILED   
3  order-service  ERROR    DEPENDENCY_FAILURE   

                                             pattern  count  \
1           Invalid password attempt for user <user>     88   
0       Upstream timeout while calling order-service     84   
2  Connection refused: could not connect to mysql...     84   
3        Order creation failed: database unavailable     84   

               first_seen               last_seen  
1 2026-10-01 00:25:42.627 2026-10-01 23:41:03.540  
0 2026-10-01 14:05:09.297 2026-10-01 14:16:55.596  
2 2026-10-01 14:05:06.207 2026-10-01 14:16:52.506  
3 2026-10-01 14:05:09.247 2026-10-01 14:16:55.546  
